# EmbeddingGemma 2: Complete Interactive Guide & Test Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**EmbeddingGemma 2** (`google/embeddinggemma-2`) is Google DeepMind's open, natively multimodal embedding model licensed under **Apache 2.0**.

### Key Highlights:
- **Unified 768-d Vector Space**: Encodes text, code, images, video, and audio into the same vector space.
- **Modular 740M Architecture**: 270M text backbone + 170M vision encoder + 300M audio encoder. You can selectively load text-only to save RAM!
- **Instruction-Tuned Asymmetric Prompts**: Optimized for `SearchQuery`, `Document`, and `CodeRetrieval`.
- **Matryoshka Representation Learning (MRL)**: Truncate embeddings to 128, 256, 512, or 768 dimensions with minimal quality loss.
- **Precision Rule**: Must use **`bfloat16`** or **`float32`** (never `float16` due to Gemma activation ranges).

## 1. Setup & Environment Installation
Run this cell to install the latest compatible libraries in Colab.

In [ ]:
!pip install -q "sentence-transformers>=6.1.0" "transformers>=4.49.0" accelerate pillow matplotlib numpy

## 2. Hardware Detection & Safe Precision Selection
On Google Colab Pro, we detect your GPU (L4, A100, or T4) and automatically select **`bfloat16`** for tensor core acceleration.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute Device: {device.upper()}")

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    bf16_ok = torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16_ok else torch.float32
    print(f"GPU: {gpu_name} (Native bfloat16: {bf16_ok})")
else:
    dtype = torch.float32
    print("CPU Mode: using float32 precision")

model_kwargs = {"torch_dtype": dtype}

## 3. Text Embeddings & Semantic Similarity Matrix
Here we load the **text-only backbone** (`~270M` parameters) by passing `config_kwargs={"vision_config": None, "audio_config": None}`. This drastically reduces RAM usage!

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt

model_id = "google/embeddinggemma-2"
print(f"Loading {model_id} (Text-Only Mode)...")

text_model = SentenceTransformer(
    model_id,
    device=device,
    model_kwargs=model_kwargs,
    config_kwargs={"vision_config": None, "audio_config": None}
)
print("Model loaded successfully!")

In [ ]:
sentences = [
    "Google DeepMind released EmbeddingGemma 2 for multimodal embeddings.",
    "EmbeddingGemma maps code, text, and images into a 768-d space.",
    "Baking chocolate chip cookies requires butter, flour, and brown sugar.",
    "Preheat the oven to 375F to bake sweet homemade pastries.",
    "Convolutional neural networks extract visual features from images."
]

# Encode into 768-d vectors
embs = text_model.encode(sentences)
print(f"Embeddings shape: {embs.shape}")

# Compute cosine similarity matrix
sim = text_model.similarity(embs, embs).cpu().numpy()

# Plot heatmap
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(sim, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(im, label="Cosine Similarity")
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels([f"Doc {i+1}" for i in range(len(sentences))])
ax.set_yticklabels([f"Doc {i+1}" for i in range(len(sentences))])
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="white" if sim[i, j] < 0.6 else "black")
plt.title("EmbeddingGemma 2 Pairwise Semantic Similarity")
plt.tight_layout()
plt.show()

## 4. Asymmetric Task Prompts (SearchQuery vs Document vs CodeRetrieval)
EmbeddingGemma 2 uses instruction prefixes for asymmetric search tasks:
- Target passages: `prompt_name="Document"` with `title: {title} | text: {content}`
- General query: `prompt_name="SearchQuery"`
- Code search: `prompt_name="CodeRetrieval"`

In [ ]:
docs = [
    {"title": "MRL Embeddings", "text": "Matryoshka Representation Learning compresses 768-d embeddings to 256-d or 128-d with high fidelity."},
    {"title": "QuickSort in Python", "text": "def quicksort(arr):\n    if len(arr) <= 1: return arr\n    pivot = arr[0]\n    return quicksort([x for x in arr[1:] if x < pivot]) + [pivot] + quicksort([x for x in arr[1:] if x >= pivot])"},
    {"title": "Espresso Extraction", "text": "Espresso requires 9 bars of pressure, 93C water, and a 1:2 coffee to water brew ratio in 25-30s."}
]

formatted_docs = [f"title: {d['title']} | text: {d['text']}" for d in docs]
doc_embeddings = text_model.encode(formatted_docs, prompt_name="Document")

queries = [
    ("how to reduce vector storage in milvus or qdrant", "SearchQuery"),
    ("python sorting algorithm with pivot", "CodeRetrieval"),
    ("ideal water pressure and temperature for making coffee", "SearchQuery")
]

for q_text, prompt_type in queries:
    q_emb = text_model.encode([q_text], prompt_name=prompt_type)
    scores = text_model.similarity(q_emb, doc_embeddings).cpu().numpy()[0]
    best_idx = np.argmax(scores)
    print(f"\nQuery: '{q_text}' [{prompt_type}]")
    print(f"Top Match: '{docs[best_idx]['title']}' (Score: {scores[best_idx]:.4f})")
    print(f"Content: {docs[best_idx]['text']}")

## 5. Matryoshka Representation Learning (MRL) Benchmark
Test how well the model preserves similarity when truncating vectors from 768d down to 512d, 256d, and 128d.

In [ ]:
dims = [768, 512, 256, 128]
test_query = "how to optimize vector databases"

# Reference 768d similarity
ref_q = text_model.encode([test_query], prompt_name="SearchQuery", truncate_dim=768)
ref_d = text_model.encode(formatted_docs, prompt_name="Document", truncate_dim=768)
ref_scores = text_model.similarity(ref_q, ref_d).cpu().numpy()[0]

fig, ax = plt.subplots(figsize=(8, 4))
x_positions = np.arange(len(dims))
top_scores = []

for dim in dims:
    q_emb = text_model.encode([test_query], prompt_name="SearchQuery", truncate_dim=dim)
    d_emb = text_model.encode(formatted_docs, prompt_name="Document", truncate_dim=dim)
    scores = text_model.similarity(q_emb, d_emb).cpu().numpy()[0]
    top_scores.append(scores[0]) # Target doc is doc 0
    savings = (1 - dim/768)*100
    print(f"Dim: {dim:3d}d | Target Score: {scores[0]:.4f} | Storage Reduction: {savings:>5.1f}%")

ax.bar([str(d) + "d" for d in dims], top_scores, color=["#4285F4", "#34A853", "#FBBC05", "#EA4335"])
ax.set_ylim(0, 1.0)
ax.set_ylabel("Similarity Score")
ax.set_title("EmbeddingGemma 2: Matryoshka Dimension vs Similarity Preservation")
for i, v in enumerate(top_scores):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Multimodal Image & Text Cross-Retrieval
In full multimodal mode, EmbeddingGemma 2 maps images directly into the 768-d space alongside text.

In [ ]:
from PIL import Image, ImageDraw

# Create synthetic image for testing
img = Image.new("RGB", (256, 256), color=(135, 206, 235))
draw = ImageDraw.Draw(img)
draw.rectangle([123, 140, 133, 256], fill=(34, 139, 34)) # Stem
draw.ellipse([73, 65, 183, 175], fill=(255, 215, 0))    # Petals
draw.ellipse([100, 92, 156, 148], fill=(101, 67, 33))   # Center

print("Loading full multimodal model (vision + text)...")
multimodal_model = SentenceTransformer(
    model_id,
    device=device,
    model_kwargs=model_kwargs
)

candidates = [
    "A blooming yellow sunflower under clear blue daylight",
    "A dark stormy ocean wave crashing against cliffs",
    "A high-tech computer motherboard with glowing circuits"
]

img_emb = multimodal_model.encode([img])
text_embs = multimodal_model.encode(candidates, prompt_name="SearchQuery")

cross_sim = multimodal_model.similarity(img_emb, text_embs).cpu().numpy()[0]

plt.figure(figsize=(3, 3))
plt.imshow(img)
plt.axis("off")
plt.title("Test Image")
plt.show()

print("\nCross-Modal Similarity Rankings:")
for rank, idx in enumerate(np.argsort(cross_sim)[::-1], start=1):
    print(f"#{rank} [Score: {cross_sim[idx]:.4f}] {candidates[idx]}")

## 7. Saving Experiments Back to GitHub from Colab
When you finish running or modifying this notebook in Google Colab Pro:
1. Go to **File** $\rightarrow$ **Save a copy in GitHub**.
2. Select your repository (`test-embedding-gemma2`) and commit message.
3. On your local machine, run `git pull` to fetch your experiment runs!